# 🚀 Activity 03: Feature Engineering (Advanced)

**Track:** Feature Engineering  
**Level:** Advanced  
**Prerequisites:** 02_feature_engineering_basic.ipynb

---

## 📖 What You Will Learn
- Polynomial features & the curse of dimensionality
- Log / power transformations for skewed data
- Principal Component Analysis (PCA) for dimensionality reduction
- Variance Inflation Factor (VIF) — detecting multicollinearity
- Target encoding (with data-leakage warning!)
- Building reusable `ColumnTransformer + Pipeline` for production

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import PolynomialFeatures, PowerTransformer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Libraries loaded')

## 📊 Dataset: California Housing

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(df.shape)
print(df.describe().round(2))

## 📐 Section 1 — Log & Power Transformations

### 🧠 Concept: Why Transform?

Linear regression assumes residuals are normally distributed. Skewed target variables violate this.

- **Log transformation:** Compresses large values, expands small ones → good for right-skewed data
- **Box-Cox / Yeo-Johnson:** Generalised power transforms that optimise normality (Yeo-Johnson works on negative values too)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Original skewed features
skewed_cols = ['AveRooms', 'AveBedrms', 'Population']
for i, col in enumerate(skewed_cols):
    axes[0, i].hist(df[col], bins=50, color='steelblue', edgecolor='white')
    axes[0, i].set_title(f'{col} — original')
    
    # Log1p transform (log1p avoids log(0))
    axes[1, i].hist(np.log1p(df[col]), bins=50, color='tomato', edgecolor='white')
    axes[1, i].set_title(f'{col} — log1p')

plt.suptitle('Before vs After Log Transformation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Yeo-Johnson (handles zero & negative values) ──────────────────────────────
pt = PowerTransformer(method='yeo-johnson', standardize=True)
df_transformed = df.copy()

for col in skewed_cols:
    df_transformed[col] = pt.fit_transform(df[[col]])

print('Skewness before:')
print(df[skewed_cols].skew().round(3))
print('\nSkewness after Yeo-Johnson:')
print(df_transformed[skewed_cols].skew().round(3))

## 🔢 Section 2 — Polynomial Features

### 🧠 Concept: When Linear Isn't Enough

If the true relationship is y = β₀ + β₁x + β₂x², a purely linear model can't fit it.
Polynomial features add x², x³, x₁x₂, etc. as explicit columns.

> **⚠️ COMMON ERROR:** Polynomial features explode with dimensionality.
> With p features and degree d, you get O(p^d) features. Always use regularisation (Ridge) with polynomial features.

| Features | Degree 2 | Degree 3 |
|---|---|---|
| 2 | 6 | 10 |
| 5 | 21 | 56 |
| 10 | 66 | 286 |

In [ ]:
# ── Demonstrate polynomial feature expansion ──────────────────────────────────
X = df[['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population']]
y = df['MedHouseVal']

for deg in [1, 2, 3]:
    pf = PolynomialFeatures(degree=deg, include_bias=False)
    X_poly = pf.fit_transform(X)
    print(f'Degree {deg}: {X.shape[1]} input features → {X_poly.shape[1]} poly features')

In [ ]:
# ── Compare: Linear vs Polynomial + Ridge ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = {}
for deg in [1, 2]:
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=1.0)),  # Ridge prevents overfitting from high-degree features
    ])
    pipe.fit(X_train, y_train)
    r2 = r2_score(y_test, pipe.predict(X_test))
    results[f'Degree {deg}'] = r2
    print(f'Degree {deg} — R²: {r2:.4f}')

print(f'\nImprovement from degree 1→2: +{(results["Degree 2"] - results["Degree 1"]):.4f}')

## 📉 Section 3 — PCA: Dimensionality Reduction

### 🧠 Concept: PCA

PCA finds new axes (principal components) that maximise variance.
- PC1 explains the most variance
- PC2 explains the second most, and is orthogonal to PC1
- Components are **linear combinations** of original features

**When to use PCA:**
- Many correlated features (multicollinear)
- Visualising high-dimensional data in 2D/3D
- Reducing training time

**When NOT to use PCA:**
- When interpretability matters (components lose original meaning)
- When features are already independent

In [ ]:
# ── PCA on housing features ───────────────────────────────────────────────────
X_full = df.drop(columns=['MedHouseVal'])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)

pca = PCA()
pca.fit(X_scaled)

# Explained variance ratio
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(explained)+1), explained, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')

axes[1].plot(range(1, len(cumulative)+1), cumulative, 'bo-')
axes[1].axhline(0.95, color='red', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

n_95 = np.argmax(cumulative >= 0.95) + 1
print(f'Components needed for 95% variance: {n_95} (out of {X_full.shape[1]})')

In [ ]:
# ── Compare: Full features vs PCA-reduced ─────────────────────────────────────
y = df['MedHouseVal']
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y, test_size=0.2, random_state=42)

# Full features
pipe_full = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
pipe_full.fit(X_tr, y_tr)
r2_full = r2_score(y_te, pipe_full.predict(X_te))

# PCA reduced
pipe_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=n_95)),
    ('lr',     LinearRegression()),
])
pipe_pca.fit(X_tr, y_tr)
r2_pca = r2_score(y_te, pipe_pca.predict(X_te))

print(f'Full features ({X_full.shape[1]})  R² = {r2_full:.4f}')
print(f'PCA reduced  ({n_95})    R² = {r2_pca:.4f}')

## 🔬 Section 4 — VIF: Detecting Multicollinearity

### 🧠 Concept: Variance Inflation Factor

**Multicollinearity** = two or more features are highly correlated with each other.

This makes regression coefficients unstable and hard to interpret.

VIF measures how much the variance of a coefficient is inflated due to collinearity:

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is the R² from regressing feature $j$ on all other features.

| VIF | Interpretation |
|---|---|
| 1 | No collinearity |
| 1–5 | Moderate |
| > 5 | High — consider removing |
| > 10 | Severe — must address |

In [ ]:
# ── Compute VIF ───────────────────────────────────────────────────────────────
X_vif = X_full.copy()
X_vif_scaled = pd.DataFrame(
    StandardScaler().fit_transform(X_vif),
    columns=X_vif.columns
)

vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [
        variance_inflation_factor(X_vif_scaled.values, i)
        for i in range(X_vif_scaled.shape[1])
    ]
}).sort_values('VIF', ascending=False)

print(vif_data.to_string(index=False))

plt.figure(figsize=(8, 4))
colors = ['tomato' if v > 5 else 'steelblue' for v in vif_data['VIF']]
plt.barh(vif_data['Feature'], vif_data['VIF'], color=colors)
plt.axvline(5, color='red', linestyle='--', label='VIF=5 threshold')
plt.axvline(10, color='darkred', linestyle=':', label='VIF=10 threshold')
plt.title('Variance Inflation Factor (VIF)')
plt.legend()
plt.tight_layout()
plt.show()

## 🎯 Section 5 — Target Encoding (with Leakage Warning!)

### 🧠 Concept: Target Encoding

Target encoding replaces a categorical value with the **mean of the target** for that category.

Example: `city='London'` → mean house price in London.

> **⚠️ DATA LEAKAGE WARNING:**  
> If you compute target means on the full dataset (including test), the test rows already 'know' the target.  
> **Always** compute target means only on training folds — use `sklearn.preprocessing.TargetEncoder` (sklearn ≥ 1.3) which does this automatically.

In [ ]:
# ── Synthetic categorical feature ─────────────────────────────────────────────
np.random.seed(42)
n = len(df)
df_te = df.copy()
df_te['city'] = np.random.choice(['London','Paris','Tokyo','NYC','Berlin'], n)
y_te = df_te['MedHouseVal']

# ❌ WRONG (leaky): compute target mean on full dataset
leaky_map = df_te.groupby('city')['MedHouseVal'].mean()
print('Leaky target encoding:')
print(leaky_map)

# ✅ CORRECT: compute only on training split
X_te_tr, X_te_ts, y_te_tr, y_te_ts = train_test_split(
    df_te[['city']], y_te, test_size=0.2, random_state=42
)
train_map = X_te_tr.join(y_te_tr).groupby('city')['MedHouseVal'].mean()
X_te_tr['city_enc'] = X_te_tr['city'].map(train_map)
X_te_ts['city_enc'] = X_te_ts['city'].map(train_map)  # use train-derived map on test

global_mean = y_te_tr.mean()  # fallback for unseen categories
X_te_ts['city_enc'] = X_te_ts['city_enc'].fillna(global_mean)
print('\n✅ Correct: target map derived from training set only')
print(train_map)

## 🏗️ Section 6 — Production-Ready ColumnTransformer Pipeline

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Full California housing with engineered features
df_prod = fetch_california_housing(as_frame=True).frame.copy()

# Add some synthetic categoricals
df_prod['ocean_prox'] = np.random.choice(['NEAR BAY','INLAND','<1H OCEAN','ISLAND'], len(df_prod))

X_prod = df_prod.drop(columns=['MedHouseVal'])
y_prod = df_prod['MedHouseVal']

num_cols = X_prod.select_dtypes(include='number').columns.tolist()
cat_cols = ['ocean_prox']

# ── Numeric sub-pipeline ──────────────────────────────────────────────────────
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('poly',    PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler',  StandardScaler()),
])

# ── Categorical sub-pipeline ──────────────────────────────────────────────────
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')),
])

# ── Combine ────────────────────────────────────────────────────────────────────
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
])

full_pipe = Pipeline([
    ('prep',  preprocessor),
    ('model', Ridge(alpha=10.0)),   # Ridge needed because poly2 creates many correlated features
])

X_tr, X_te, y_tr, y_te = train_test_split(X_prod, y_prod, test_size=0.2, random_state=42)
full_pipe.fit(X_tr, y_tr)

r2 = r2_score(y_te, full_pipe.predict(X_te))
cv_scores = cross_val_score(full_pipe, X_prod, y_prod, cv=5, scoring='r2')

print(f'Test R²          : {r2:.4f}')
print(f'CV R² (5-fold)   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 🎯 Summary

| Technique | Benefit | Watch-out |
|---|---|---|
| Log / Power transform | Fixes skew; stabilises variance | Apply to positive-only data (log1p for zeros) |
| Polynomial features | Captures non-linear relationships | Dimensionality explosion — use Ridge |
| PCA | Removes multicollinearity; visualises data | Loses interpretability |
| VIF | Detects multicollinearity numerically | VIF > 5-10 = remove or combine features |
| Target encoding | Handles high-cardinality categories | Data leakage — always fit on train only |
| ColumnTransformer + Pipeline | Production-safe, leakage-free | Design sub-pipelines first, combine last |

---

**Next:** [04_linear_regression_sklearn.ipynb](04_linear_regression_sklearn.ipynb)